In [2]:
!mamba install pandas scikit-learn
import numpy as np
import pandas as pd

def optimize_fico_buckets(file_path: str, num_buckets: int = 10) -> list:
    """
    Optimizes FICO score bucket boundaries using Dynamic Programming 
    by maximizing the log-likelihood function of defaults.
    """
    # 1. Load data and aggregate defaults/counts by individual FICO score
    # Expected columns: 'fico_score' (300-850) and 'default' (0 or 1)
    df = pd.read_csv(file_path)
    
    # Pre-aggregate data for faster performance
    min_fico, max_fico = 300, 850
    fico_range = np.arange(min_fico, max_fico + 1)
    
    counts = df.groupby('fico_score')['default'].count().reindex(fico_range, fill_value=0).values
    defaults = df.groupby('fico_score')['default'].sum().reindex(fico_range, fill_value=0).values
    
    num_scores = len(fico_range)
    
    # 2. Precompute single-bucket log-likelihood contributions for every possible interval (i, j)
    # ll_matrix[i, j] represents the log-likelihood of a bucket containing FICO scores from index i to j
    ll_matrix = np.full((num_scores, num_scores), -np.inf)
    
    for i in range(num_scores):
        running_n = 0
        running_k = 0
        for j in range(i, num_scores):
            running_n += counts[j]
            running_k += defaults[j]
            
            if running_n > 0:
                if running_k == 0 or running_k == running_n:
                    ll_matrix[i, j] = 0.0  # Perfect classification has no entropy
                else:
                    p = running_k / running_n
                    ll_matrix[i, j] = running_k * np.log(p) + (running_n - running_k) * np.log(1 - p)

    # 3. Dynamic Programming execution
    # dp[b, j] stores the max log-likelihood using 'b' buckets up to FICO index 'j'
    dp = np.full((num_buckets + 1, num_scores), -np.inf)
    parent = np.zeros((num_buckets + 1, num_scores), dtype=int)
    
    # Initialize baseline for the first bucket (b = 1)
    for j in range(num_scores):
        dp[1, j] = ll_matrix[0, j]
        parent[1, j] = 0

    # Fill DP table for subsequent buckets
    for b in range(2, num_buckets + 1):
        for j in range(b - 1, num_scores):
            for i in range(b - 2, j):
                current_ll = dp[b - 1, i] + ll_matrix[i + 1, j]
                if current_ll > dp[b, j]:
                    dp[b, j] = current_ll
                    parent[b, j] = i + 1

    # 4. Backtrack to extract the optimal boundaries
    boundaries_idx = []
    curr_idx = num_scores - 1
    for b in range(num_buckets, 0, -1):
        start_idx = parent[b, curr_idx]
        boundaries_idx.append(start_idx)
        curr_idx = start_idx - 1
        
    boundaries_idx.reverse()
    
    # Translate internal indexes back to physical FICO score values
    fico_boundaries = [min_fico] + [fico_range[idx] for idx in boundaries_idx if idx > 0] + [max_fico]
    fico_boundaries = sorted(list(set(fico_boundaries))) # Remove any duplicates
    
    return fico_boundaries

def assign_ratings_map(fico_score: int, boundaries: list) -> int:
    """
    Maps a raw FICO score to a categorical integer rating based on boundaries.
    Lower ratings signify better creditworthiness (highest FICO bucket = Rating 1).
    """
    num_buckets = len(boundaries) - 1
    for i in range(num_buckets):
        # Boundaries are sorted. Better credit (higher FICO) gets lower rating index.
        if boundaries[i] <= fico_score <= boundaries[i+1]:
            return num_buckets - i  # Inverts rating so highest bucket gets Rating 1
    return num_buckets

# --- Validation Script Execution Example ---
if __name__ == "__main__":
    DATA_PATH = "Task 3 and 4_Loan_Data.csv"
    
    try:
        # Calculate optimal boundaries for 10 buckets
        optimal_boundaries = optimize_fico_buckets(DATA_PATH, num_buckets=10)
        
        print("--- Optimal FICO Quantization Map ---")
        print(f"Calculated Bucket Boundaries: {optimal_boundaries}\n")
        
        # Display sample mapping matrix for Charlie's categorization model
        sample_scores = [320, 550, 620, 710, 790, 840]
        print("--- Sample Rating Conversions (Lower Rating = Safer Borrower) ---")
        for score in sample_scores:
            rating = assign_ratings_map(score, optimal_boundaries)
            print(f"Raw FICO Score: {score} ---> Assigned Categorical Rating: {rating}")
            
    except FileNotFoundError:
        print(f"Please place your mortgage history dataset in the path as '{DATA_PATH}' to execute dynamic programming optimization.")
import numpy as np
import pandas as pd

def optimize_fico_buckets(file_path: str, num_buckets: int = 10) -> list:
    """
    Optimizes FICO score bucket boundaries using Dynamic Programming 
    by maximizing the log-likelihood function of defaults.
    """
    # 1. Load data and aggregate defaults/counts by individual FICO score
    # Expected columns: 'fico_score' (300-850) and 'default' (0 or 1)
    df = pd.read_csv(file_path)
    
    # Pre-aggregate data for faster performance
    min_fico, max_fico = 300, 850
    fico_range = np.arange(min_fico, max_fico + 1)
    
    counts = df.groupby('fico_score')['default'].count().reindex(fico_range, fill_value=0).values
    defaults = df.groupby('fico_score')['default'].sum().reindex(fico_range, fill_value=0).values
    
    num_scores = len(fico_range)
    
    # 2. Precompute single-bucket log-likelihood contributions for every possible interval (i, j)
    # ll_matrix[i, j] represents the log-likelihood of a bucket containing FICO scores from index i to j
    ll_matrix = np.full((num_scores, num_scores), -np.inf)
    
    for i in range(num_scores):
        running_n = 0
        running_k = 0
        for j in range(i, num_scores):
            running_n += counts[j]
            running_k += defaults[j]
            
            if running_n > 0:
                if running_k == 0 or running_k == running_n:
                    ll_matrix[i, j] = 0.0  # Perfect classification has no entropy
                else:
                    p = running_k / running_n
                    ll_matrix[i, j] = running_k * np.log(p) + (running_n - running_k) * np.log(1 - p)

    # 3. Dynamic Programming execution
    # dp[b, j] stores the max log-likelihood using 'b' buckets up to FICO index 'j'
    dp = np.full((num_buckets + 1, num_scores), -np.inf)
    parent = np.zeros((num_buckets + 1, num_scores), dtype=int)
    
    # Initialize baseline for the first bucket (b = 1)
    for j in range(num_scores):
        dp[1, j] = ll_matrix[0, j]
        parent[1, j] = 0

    # Fill DP table for subsequent buckets
    for b in range(2, num_buckets + 1):
        for j in range(b - 1, num_scores):
            for i in range(b - 2, j):
                current_ll = dp[b - 1, i] + ll_matrix[i + 1, j]
                if current_ll > dp[b, j]:
                    dp[b, j] = current_ll
                    parent[b, j] = i + 1

    # 4. Backtrack to extract the optimal boundaries
    boundaries_idx = []
    curr_idx = num_scores - 1
    for b in range(num_buckets, 0, -1):
        start_idx = parent[b, curr_idx]
        boundaries_idx.append(start_idx)
        curr_idx = start_idx - 1
        
    boundaries_idx.reverse()
    
    # Translate internal indexes back to physical FICO score values
    fico_boundaries = [min_fico] + [fico_range[idx] for idx in boundaries_idx if idx > 0] + [max_fico]
    fico_boundaries = sorted(list(set(fico_boundaries))) # Remove any duplicates
    
    return fico_boundaries

def assign_ratings_map(fico_score: int, boundaries: list) -> int:
    """
    Maps a raw FICO score to a categorical integer rating based on boundaries.
    Lower ratings signify better creditworthiness (highest FICO bucket = Rating 1).
    """
    num_buckets = len(boundaries) - 1
    for i in range(num_buckets):
        # Boundaries are sorted. Better credit (higher FICO) gets lower rating index.
        if boundaries[i] <= fico_score <= boundaries[i+1]:
            return num_buckets - i  # Inverts rating so highest bucket gets Rating 1
    return num_buckets

# --- Validation Script Execution Example ---
if __name__ == "__main__":
    DATA_PATH = "Task 3 and 4_Loan_Data.csv"
    
    try:
        # Calculate optimal boundaries for 10 buckets
        optimal_boundaries = optimize_fico_buckets(DATA_PATH, num_buckets=10)
        
        print("--- Optimal FICO Quantization Map ---")
        print(f"Calculated Bucket Boundaries: {optimal_boundaries}\n")
        
        # Display sample mapping matrix for Charlie's categorization model
        sample_scores = [320, 550, 620, 710, 790, 840]
        print("--- Sample Rating Conversions (Lower Rating = Safer Borrower) ---")
        for score in sample_scores:
            rating = assign_ratings_map(score, optimal_boundaries)
            print(f"Raw FICO Score: {score} ---> Assigned Categorical Rating: {rating}")
            
    except FileNotFoundError:
        print(f"Please place your mortgage history dataset in the path as '{DATA_PATH}' to execute dynamic programming optimization.")


mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.292 seconds
All requested packages already installed.
--- Optimal FICO Quantization Map ---
Calculated Bucket Boundaries: [300, np.int32(521), np.int32(553), np.int32(581), np.int32(612), np.int32(650), np.int32(697), np.int32(733), np.int32(753), np.int32(754), 850]

--- Sample Rating Conversions (Lower Rating = Safer Borrower) ---
Raw FICO Score: 320 ---> Assigned Categorical Rating: 10
Raw FICO Score: 550 ---> Assigned Categorical Rating: 9
Raw FICO Score: 620 ---> Assigned Categorical Rating: 6
Raw FICO Score: 710 ---> Assigned Categorical Rating: 4
Raw FICO Score: 790 ---> Assigned Categorical Rating: 1
Raw FICO Score: 840 ---> Assigned Categorical Rating: 1
--- Optimal FICO Quantization Map ---
Calculated Bucket Boundaries: [300, np.int32(521), np.int32(553), np.int32(581), np.in